# 04 — Forecasting Tutorial

**Phase 3, Week 9** — CMU-Africa Techskills educational notebook on energy load forecasting.

**Goal:** Predict future electricity consumption using our **cleaned, interpolated** Phase 2 dataset — a continuous 30-minute smart-meter timeline where anomalous intervals were repaired so forecasting models see a stable series, not raw spikes and gaps.

**Scope (this notebook):** We start from the production clean CSV, then (in later sections) learn why time series must be split **chronologically**, and how **lag features** turn a sequence into supervised tabular data for models like XGBoost.

## 1. Setup & Load Clean Data

**Why:** Forecasting in this project starts from the same production artifact as our Phase 3 scripts — `data/processed/clean_smart_meter_data.csv` written by `scripts/generate_clean_data.py`. Reusing that file keeps the tutorial aligned with the research pipeline and avoids retraining Isolation Forest inside the notebook.

If the CSV is missing locally, regenerate it from the repository root:

```bash
python scripts/generate_clean_data.py
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
CLEAN_CSV = PROJECT_ROOT / "data" / "processed" / "clean_smart_meter_data.csv"

df = pd.read_csv(CLEAN_CSV, parse_dates=["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

print(f"Loaded: {CLEAN_CSV}")
print(f"Shape: {df.shape}")
print(f"Timestamp range: {df['Timestamp'].iloc[0]} -> {df['Timestamp'].iloc[-1]}")
print(f"Electricity_Consumed NaNs: {df['Electricity_Consumed'].isna().sum()}")
df.head()

## 2. Chronological Splitting (Avoiding Data Leakage)

**Why this matters:** In ordinary machine learning we often shuffle rows and split at random. That is **unsafe for time series**.

**Data leakage** happens when information from the future sneaks into the training set. If a random split puts Tuesday’s afternoon reading in train and Monday morning in test, the model has already “seen” the future when it learns patterns — and test scores look unrealistically good.

**Chronological splitting** keeps time in order:
- **Train** — earliest 70% of the timeline (learn the pattern)
- **Validation** — next 15% (tune / check while developing)
- **Test** — final 15% (honest evaluation on the future)

This matches the Phase 3 research protocol used in `time_series_split` and our forecasting scripts.

In [ ]:
TARGET_COLUMN = "Electricity_Consumed"
TRAIN_PCT = 0.70
VAL_PCT = 0.15

n = len(df)
train_end = int(n * TRAIN_PCT)
val_end = int(n * (TRAIN_PCT + VAL_PCT))

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

for name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
    print(
        f"{name:5s}: rows={len(frame):4d} | "
        f"{frame['Timestamp'].iloc[0]} -> {frame['Timestamp'].iloc[-1]}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(
    train_df["Timestamp"],
    train_df[TARGET_COLUMN],
    color="tab:blue",
    label="Train (70%)",
    linewidth=1,
)
ax.plot(
    val_df["Timestamp"],
    val_df[TARGET_COLUMN],
    color="tab:orange",
    label="Validation (15%)",
    linewidth=1,
)
ax.plot(
    test_df["Timestamp"],
    test_df[TARGET_COLUMN],
    color="tab:green",
    label="Test (15%)",
    linewidth=1,
)

ax.set_title("Chronological train / validation / test split")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Electricity_Consumed")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3. Lag Features (Turning Time into a Table)

**Why:** Models like XGBoost do not “see” time the way humans do. They need each row to look like a normal supervised learning example: **features in, target out**.

A **lag feature** is a past value of the target (or another series) aligned with the current row. To predict consumption at time $t$, we can use:
- **`lag_1`** — value at $t-1$ (previous 30-minute interval)
- **`lag_48`** — value at $t-48$ (same time of day yesterday, because $48 \times 30\text{ min} = 24\text{ h}$)

Below we take a small slice of the **training** set and build those lags with an explicit `shift`, so you can see the numbers move sideways next to the original target.

In [ ]:
# Small chronological window from the start of train (enough history for lag_48)
demo = train_df[["Timestamp", TARGET_COLUMN]].iloc[:80].copy()
demo["lag_1"] = demo[TARGET_COLUMN].shift(1)
demo["lag_48"] = demo[TARGET_COLUMN].shift(48)

# Show rows where both lags exist so the shift is easy to verify by eye
lag_demo = demo.dropna(subset=["lag_1", "lag_48"]).head(10)

print("Compare each target at time t with lag_1 (t-1) and lag_48 (t-48):")
lag_demo[["Timestamp", TARGET_COLUMN, "lag_1", "lag_48"]]

## 4. Training an XGBoost Forecaster

**Why XGBoost here?** Statistical baselines (seasonal naive, Prophet) mainly capture smooth seasonality and trend. They are strong floors, but they do not automatically exploit the rich **tabular** predictors we already engineered — lag history, weather, and calendar fields.

**XGBoost** (extreme gradient boosting) builds many shallow decision trees in sequence. Each new tree focuses on the mistakes of the previous ones. On a supervised table, that lets the model learn **non-linear interactions** among:
- consumption lags (`lag_1`, `lag_2`, `lag_48` — recent and same-time-yesterday memory)
- weather (`Temperature`, `Humidity`)
- calendar context (`hour`, `day_of_week`, `month`, `is_weekend`)

Below we rebuild the full supervised lag matrix on the clean series (same helper as our Phase 3 scripts), re-split chronologically, then initialize and fit an `XGBRegressor`.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from xgboost import XGBRegressor

from src.features.build_features import create_supervised_lags

FEATURE_COLUMNS = [
    f"{TARGET_COLUMN}_lag_1",
    f"{TARGET_COLUMN}_lag_2",
    f"{TARGET_COLUMN}_lag_48",
    "Temperature",
    "Humidity",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
]

# Full-series lags, then chronological split (matches evaluate_xgboost.py)
tabular_df = create_supervised_lags(df, target_col=TARGET_COLUMN)
n_tab = len(tabular_df)
train_end_tab = int(n_tab * TRAIN_PCT)
val_end_tab = int(n_tab * (TRAIN_PCT + VAL_PCT))

xgb_train = tabular_df.iloc[:train_end_tab]
xgb_val = tabular_df.iloc[train_end_tab:val_end_tab]
xgb_test = tabular_df.iloc[val_end_tab:]

X_train = xgb_train[FEATURE_COLUMNS]
y_train = xgb_train[TARGET_COLUMN]
X_val = xgb_val[FEATURE_COLUMNS]
y_val = xgb_val[TARGET_COLUMN]

model = XGBRegressor(n_estimators=100, learning_rate=0.1)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print(
    f"Trained XGBRegressor on {len(X_train)} rows "
    f"({len(FEATURE_COLUMNS)} features); val size={len(X_val)}"
)
print(f"Held-out test rows (for later evaluation): {len(xgb_test)}")

## 5. Scoring the Forecast (MAE & RMSE)

Training alone does not tell us if the model is useful. We score predictions on the **held-out test** window — the future the model never saw during `fit`.

Two standard metrics (same helpers as our Phase 3 scripts):

- **MAE (Mean Absolute Error)** — the average absolute miss. Easy to explain: “on average, how far off (in consumption units) is each prediction?” Every error counts equally.
- **RMSE (Root Mean Squared Error)** — like MAE, but **large mistakes hurt more** because errors are squared before averaging. If RMSE is much higher than MAE, a few big misses are dragging the score down.

Lower is better for both. We compare against research floors later; here we print the test MAE and RMSE for this XGBoost run.

In [ ]:
from src.models.evaluate_forecast import (
    mean_absolute_error_forecast,
    root_mean_squared_error_forecast,
)

X_test = xgb_test[FEATURE_COLUMNS]
y_test = xgb_test[TARGET_COLUMN]
y_pred = model.predict(X_test)

mae = mean_absolute_error_forecast(y_test, y_pred)
rmse = root_mean_squared_error_forecast(y_test, y_pred)

print(f"Test predictions: {len(y_pred)} rows")
print(f"MAE:  {mae:.6f}")
print(f"RMSE: {rmse:.6f}")

In [ ]:
# ~3 days at 30-minute resolution (same window length as compare_forecasts.py)
PLOT_WINDOW_STEPS = 144

plot_ts = xgb_test["Timestamp"].iloc[:PLOT_WINDOW_STEPS].reset_index(drop=True)
plot_actual = y_test.iloc[:PLOT_WINDOW_STEPS].to_numpy()
plot_pred = y_pred[:PLOT_WINDOW_STEPS]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(plot_ts, plot_actual, color="tab:blue", label="Actual", linewidth=1.5)
ax.plot(
    plot_ts,
    plot_pred,
    color="tab:red",
    linestyle="--",
    label="Predicted",
    linewidth=1.5,
)
ax.set_title("XGBoost forecast — first ~3 days of the test window")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Electricity_Consumed")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. What This Notebook Achieved

In this tutorial you walked the core Phase 3 forecasting path:

1. **Started from clean data** — the interpolated production CSV, so models see a continuous 30-minute timeline.
2. **Split chronologically** — train → validation → test in time order to avoid leaking the future into training.
3. **Built lag features** — turned the series into a supervised table (`lag_1`, `lag_48`, …) that tree models can learn from.
4. **Trained XGBoost** — gradient boosting on lag, weather, and calendar features.
5. **Scored and visualized** — MAE / RMSE on the held-out test set, plus an Actual vs Predicted chart over ~3 days.

From here, explore the research ladder in the repo (`naive` → Prophet → XGBoost → LSTM via `scripts/compare_forecasts.py`) or the end-to-end CLI (`python main.py --model xgboost`).